<a href="https://colab.research.google.com/github/Teewodros/Sorting_Algorithms_DSA_Project/blob/main/LEX_and_YACC_Compiler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [1]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 2 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 [10.7 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl-dev amd64 2.6.4-8build2 [6,236 B]
Fetched 1,072 kB in 1s (888 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot

## Lex only

In [31]:
#@title Writing Lex program
%%writefile program.l


WORD [^ \t\n,\.:]+
EOL [\n]
BLANK [ ]
%%
ask         {printf("TOKEN ASK\n");}
say         {printf("TOKEN SAY\n");}
if          {printf("TOKEN IF\n");}
else        {printf("TOKEN ELSE\n");}
end         {printf("TOKEN END\n");}
repeat      {printf("TOKEN REPEAT\n");}
while       {printf("TOKEN WHILE\n");}

int         {printf("TOKEN TYPE_INT\n");}
real        {printf("TOKEN TYPE_REAL\n");}
string      {printf("TOKEN TYPE_STRING\n");}

[a-zA-Z][a-zA-Z0-9]*   {printf("TOKEN IDENTIFIER: %s\n", yytext);}

":"        {printf("TOKEN COLON\n");}

"<-"       {printf("TOKEN ASSIGN\n");}

">="       {printf("TOKEN GREATER_EQUAL\n");}
"<="       {printf("TOKEN LESS_EQUAL\n");}
">"        {printf("TOKEN GREATER\n");}
"<"        {printf("TOKEN LESS\n");}
"="        {printf("TOKEN EQUAL\n");}

"+"        {printf("TOKEN PLUS\n");}
"-"        {printf("TOKEN MINUS\n");}
"*"        {printf("TOKEN MULTIPLY\n");}
"/"        {printf("TOKEN DIVIDE\n");}


[0-9]+"."[0-9]+    {printf("TOKEN REAL_LITERAL: %s\n", yytext);}
[0-9]+             {printf("TOKEN INTEGER_LITERAL: %s\n", yytext);}

\"[^\n"]*\"        {printf("TOKEN STRING_LITERAL: %s\n", yytext);}

[ \t\n] ;

. {printf("LEXICAL ERROR: %s\n", yytext);}
%%

void main(int argc, char *argv[]){
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }

    yyin=fopen(argv[1],"r");
    yylex();


    fclose(yyin);

}

int yywrap(){
    return 1;
}

Overwriting program.l


In [64]:
#@title Writing Lex program
%%writefile program.l
%{
#include <stdio.h>
#include <stdlib.h>

/* counters (like the slide) */
int tokenCount=0;

int kwCount=0, typeCount=0, idCount=0;
int intLitCount=0, realLitCount=0, strLitCount=0;
int assignCount=0, arithCount=0, relCount=0, colonCount=0;
int errCount=0;
%}

%%

"#".*   ;

ask         { tokenCount++; kwCount++;    printf("Token %d: KEYWORD (%s)\n", tokenCount, yytext); }
say         { tokenCount++; kwCount++;    printf("Token %d: KEYWORD (%s)\n", tokenCount, yytext); }
if          { tokenCount++; kwCount++;    printf("Token %d: KEYWORD (%s)\n", tokenCount, yytext); }
else        { tokenCount++; kwCount++;    printf("Token %d: KEYWORD (%s)\n", tokenCount, yytext); }
end         { tokenCount++; kwCount++;    printf("Token %d: KEYWORD (%s)\n", tokenCount, yytext); }
repeat      { tokenCount++; kwCount++;    printf("Token %d: KEYWORD (%s)\n", tokenCount, yytext); }
while       { tokenCount++; kwCount++;    printf("Token %d: KEYWORD (%s)\n", tokenCount, yytext); }

int         { tokenCount++; typeCount++;  printf("Token %d: TYPE (%s)\n", tokenCount, yytext); }
real        { tokenCount++; typeCount++;  printf("Token %d: TYPE (%s)\n", tokenCount, yytext); }
string      { tokenCount++; typeCount++;  printf("Token %d: TYPE (%s)\n", tokenCount, yytext); }

[a-zA-Z][a-zA-Z0-9]*   { tokenCount++; idCount++; printf("Token %d: IDENTIFIER (%s)\n", tokenCount, yytext); }

":"        { tokenCount++; colonCount++;  printf("Token %d: COLON (:)\n", tokenCount); }

"<-"       { tokenCount++; assignCount++; printf("Token %d: ASSIGN (<-)\n", tokenCount); }

">="       { tokenCount++; relCount++;    printf("Token %d: REL_OP (>=)\n", tokenCount); }
"<="       { tokenCount++; relCount++;    printf("Token %d: REL_OP (<=)\n", tokenCount); }
">"        { tokenCount++; relCount++;    printf("Token %d: REL_OP (>)\n", tokenCount); }
"<"        { tokenCount++; relCount++;    printf("Token %d: REL_OP (<)\n", tokenCount); }
"="        { tokenCount++; relCount++;    printf("Token %d: REL_OP (=)\n", tokenCount); }

"+"        { tokenCount++; arithCount++;  printf("Token %d: OPERATOR (+)\n", tokenCount); }
"-"        { tokenCount++; arithCount++;  printf("Token %d: OPERATOR (-)\n", tokenCount); }
"*"        { tokenCount++; arithCount++;  printf("Token %d: OPERATOR (*)\n", tokenCount); }
"/"        { tokenCount++; arithCount++;  printf("Token %d: OPERATOR (/)\n", tokenCount); }

[0-9]+"."[0-9]+    { tokenCount++; realLitCount++; printf("Token %d: REAL (%s)\n", tokenCount, yytext); }
[0-9]+             { tokenCount++; intLitCount++;  printf("Token %d: INTEGER (%s)\n", tokenCount, yytext); }

\"[^\n"]*\"        { tokenCount++; strLitCount++;  printf("Token %d: STRING (%s)\n", tokenCount, yytext); }

[ \t\n\r]+ ;

. { tokenCount++; errCount++; printf("Token %d: LEXICAL_ERROR (%s)\n", tokenCount, yytext); }

%%

void main(int argc, char *argv[]){
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }

    yyin=fopen(argv[1],"r");
    if(!yyin){
        printf("Cannot open file: %s\n", argv[1]);
        exit(0);
    }

    yylex();
    fclose(yyin);

    printf("\n--------------------------------\n");
    printf("Total tokens: %d\n\n", tokenCount);

    printf("Keywords                 : %d\n", kwCount);
    printf("Types                    : %d\n", typeCount);
    printf("Identifiers              : %d\n", idCount);
    printf("Integer Literals         : %d\n", intLitCount);
    printf("Real Literals            : %d\n", realLitCount);
    printf("String Literals          : %d\n", strLitCount);
    printf("Assignment Operators (<-): %d\n", assignCount);
    printf("Arithmetic Operators     : %d\n", arithCount);
    printf("Relational Operators     : %d\n", relCount);
    printf("Colons (:)               : %d\n", colonCount);
    if(errCount>0) printf("Lexical Errors           : %d\n", errCount);
}

int yywrap(){
    return 1;
}

Overwriting program.l


In [52]:
#@title Writing Lex program
%%writefile program.l



%%
"# ".*   ;
ask         {printf("TOKEN ASK\n");}
say         {printf("TOKEN SAY\n");}
if          {printf("TOKEN IF\n");}
else        {printf("TOKEN ELSE\n");}
end         {printf("TOKEN END\n");}
repeat      {printf("TOKEN REPEAT\n");}
while       {printf("TOKEN WHILE\n");}

int         {printf("TOKEN TYPE_INT\n");}
real        {printf("TOKEN TYPE_REAL\n");}
string      {printf("TOKEN TYPE_STRING\n");}

[a-zA-Z][a-zA-Z0-9]*   {printf("IDENTIFIER (%s)\n", yytext);}
"<-"       {printf("TOKEN ASSIGN\n");}

">="       {printf("TOKEN GREATER_EQUAL\n");}
"<="       {printf("TOKEN LESS_EQUAL\n");}
">"        {printf("TOKEN GREATER\n");}
"<"        {printf("TOKEN LESS\n");}
"="        {printf("TOKEN EQUAL\n");}

"+"        {printf("TOKEN PLUS\n");}
"-"        {printf("TOKEN MINUS\n");}
"*"        {printf("TOKEN MULTIPLY\n");}
"/"        {printf("TOKEN DIVIDE\n");}


[0-9]+"."[0-9]+    {printf("TOKEN REAL_LITERAL: %s\n", yytext);}
[0-9]+             {printf("TOKEN INTEGER_LITERAL: %s\n", yytext);}

\"[^\n"]*\"        {printf("TOKEN STRING_LITERAL: %s\n", yytext);}

[ \t\n] ;

. {printf("LEXICAL ERROR: %s\n", yytext);}
%%

void main(int argc, char *argv[]){
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }

    yyin=fopen(argv[1],"r");
    yylex();


    fclose(yyin);

}

int yywrap(){
    return 1;
}

Overwriting program.l


if you want to use at txt as an input

In [36]:
%%writefile program.txt

# Declaration
a : int
b : int
sum : int
i : int

# Input
say "Enter a and b: "
ask a
ask b

# Calculation
sum <- a + b

# Condition
if sum >= 0
 say "Sum is non-negative"
else
 say "Sum is negative"
end

# Loop
i <- 0
repeat while i < sum
 say i
 i <- i + 1
end

Overwriting program.txt


In [65]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex -l program.l
gcc lex.yy.c
./a.out program.txt

Token 1: IDENTIFIER (a)
Token 2: COLON (:)
Token 3: TYPE (int)
Token 4: IDENTIFIER (b)
Token 5: COLON (:)
Token 6: TYPE (int)
Token 7: IDENTIFIER (sum)
Token 8: COLON (:)
Token 9: TYPE (int)
Token 10: IDENTIFIER (i)
Token 11: COLON (:)
Token 12: TYPE (int)
Token 13: KEYWORD (say)
Token 14: STRING ("Enter a and b: ")
Token 15: KEYWORD (ask)
Token 16: IDENTIFIER (a)
Token 17: KEYWORD (ask)
Token 18: IDENTIFIER (b)
Token 19: IDENTIFIER (sum)
Token 20: ASSIGN (<-)
Token 21: IDENTIFIER (a)
Token 22: OPERATOR (+)
Token 23: IDENTIFIER (b)
Token 24: KEYWORD (if)
Token 25: IDENTIFIER (sum)
Token 26: REL_OP (>=)
Token 27: INTEGER (0)
Token 28: KEYWORD (say)
Token 29: STRING ("Sum is non-negative")
Token 30: KEYWORD (else)
Token 31: KEYWORD (say)
Token 32: STRING ("Sum is negative")
Token 33: KEYWORD (end)
Token 34: IDENTIFIER (i)
Token 35: ASSIGN (<-)
Token 36: INTEGER (0)
Token 37: KEYWORD (repeat)
Token 38: KEYWORD (while)
Token 39: IDENTIFIER (i)
Token 40: REL_OP (<)
Token 41: IDENTIFIER (sum

## Lex and Yacc combined

In [ ]:
#@title Writing YACC program
%%writefile program.y

%{
    #include<stdio.h>
    #include<stdlib.h>
%}
%token DIGIT LETTER UND NL
%%
stmt: variable NL {printf("Valid Identifier\n");exit(0);}
variable: LETTER alphanumeric;
alphanumeric: LETTER alphanumeric
            | DIGIT alphanumeric
            | UND alphanumeric
            | LETTER
            | DIGIT
            | UND;
%%

int yyerror(){
    printf("Invalid Identifier\n");
    exit(0);
}

void main(){
    printf("Enter the variable name: ");
    yyparse();
}

Overwriting program.y


In [ ]:
#@title Writing Lex program
%%writefile program.l

%{
    #include "y.tab.h"
%}
%%
[a-zA-Z] {return LETTER;}
[0-9] {return DIGIT;}
[_] {return UND;}
\n {return NL;}
. {return yytext[0];}
%%

Overwriting program.l


if you want to use at txt as an input

In [ ]:
%%writefile program.txt

This is a sample file.

In [ ]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

yacc -d program.y
lex program.l
cc y.tab.c lex.yy.c -ll
./a.out

y.tab.c: In function ‘yyparse’:
y.tab.c:1121:16: warning: implicit declaration of function ‘yylex’ [-Wimplicit-function-declaration]
       yychar = yylex ();
                ^~~~~
y.tab.c:1256:7: warning: implicit declaration of function ‘yyerror’; did you mean ‘yyerrok’? [-Wimplicit-function-declaration]
       yyerror (YY_("syntax error"));
       ^~~~~~~
       yyerrok
Enter the variable name: variable_name
Valid Identifier
